# Diagnostics to CSV

This notebook runs full folder diagnostics and cross-folder comparisons
and writes CSV outputs instead of long text reports.

Edit the paths and compare specs in the next cell, then run all cells.

In [26]:
from pathlib import Path

# Base folder with person_### subfolders
BASE_DIR = Path("./outputs/simplified_test_reid_088/refined")

# Where to write CSV outputs
OUTPUT_DIR = Path("./outputs/diagnostics_csv_v4")

# Compare specs (same format as compare_folders_deep.py, but a list)
COMPARE_SPECS = [
]

# Optional: limit images per folder for faster runs (None for all)
MAX_IMAGES_PER_FOLDER = None

# Skip top-pair image similarity if too many pairs
MAX_PAIRWISE = 2_000_000

# If True, use LARGE_* thresholds when either folder is large
USE_LARGE_THRESHOLDS = False

In [27]:
import csv
import re
import sys
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import onnxruntime as ort

ROOT = Path.cwd()
if (ROOT / "config.py").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT / "reid_system" / "config.py").exists():
    sys.path.insert(0, str(ROOT / "reid_system"))

import config

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
FILENAME_RE = re.compile(r"frame_(\d+).*_l(\d+)")
PAIR_SPEC_RE = re.compile(r"\s*(.*?)\s*(?:\bis\b|:|=)\s*(.*?)\s*$", re.IGNORECASE)

def write_csv(rows: List[dict], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fieldnames = sorted({k for row in rows for k in row.keys()})
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def load_osnet():
    sess = ort.InferenceSession(str(config.OSNET_MODEL), providers=["CPUExecutionProvider"])
    iname = sess.get_inputs()[0].name
    return sess, iname

def preprocess_img(img, size=(256, 128)):
    h, w = size
    img = cv2.resize(img, (w, h))
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = np.transpose(rgb, (2, 0, 1))
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)
    return (rgb - mean) / std

def extract_emb(img, sess, iname):
    tensor = preprocess_img(img)[np.newaxis, ...]
    emb = sess.run(None, {iname: tensor})[0][0]
    return emb / (np.linalg.norm(emb) + 1e-12)

def extract_color_sig(img):
    resized = cv2.resize(img, (64, 128))
    lab = cv2.cvtColor(resized, cv2.COLOR_BGR2LAB)
    split = lab.shape[0] // 2
    upper = lab[:split, :, :]
    lower = lab[split:, :, :]
    hist_u = cv2.calcHist([upper], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256]).flatten().astype(np.float32)
    hist_l = cv2.calcHist([lower], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256]).flatten().astype(np.float32)
    hist_u /= (hist_u.sum() + 1e-12)
    hist_l /= (hist_l.sum() + 1e-12)
    return hist_u, hist_l

def color_similarity(u1, l1, u2, l2):
    up = float(np.minimum(u1, u2).sum())
    lo = float(np.minimum(l1, l2).sum())
    return 0.4 * up + 0.6 * lo

def resolve_folder_token(root: Path, token: str) -> Path:
    token = token.strip()
    if not token:
        raise ValueError("Empty folder token in comparison spec")
    path = Path(token)
    if path.is_absolute():
        return path
    if path.exists():
        return path
    if token.isdigit():
        return root / f"person_{int(token):03d}"
    if token.startswith("person_"):
        return root / token
    return root / token

def parse_pair_spec(root: Path, spec: str) -> List[Tuple[Path, Path]]:
    match = PAIR_SPEC_RE.match(spec)
    if not match:
        raise ValueError(f"Could not parse comparison spec: {spec!r}")
    left_text, right_text = match.group(1), match.group(2)
    left_tokens = [token.strip() for token in left_text.split(",") if token.strip()]
    right_tokens = [token.strip() for token in right_text.split(",") if token.strip()]
    if len(right_tokens) != 1:
        raise ValueError(f"Right-hand side must contain exactly one folder: {spec!r}")
    right_folder = resolve_folder_token(root, right_tokens[0])
    return [(resolve_folder_token(root, token), right_folder) for token in left_tokens]

def list_images(folder: Path, max_images: Optional[int]):
    images = sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXT])
    if max_images and len(images) > max_images:
        idx = np.linspace(0, len(images) - 1, max_images, dtype=int)
        images = [images[i] for i in idx]
    return images

def summarize_tracks(track_buckets: Dict[int, dict]):
    track_protos = {}
    track_stats = {}
    for tid, data in track_buckets.items():
        if not data["embs"]:
            continue
        embs = np.vstack(data["embs"])
        proto_emb = embs.mean(axis=0)
        proto_emb /= (np.linalg.norm(proto_emb) + 1e-12)
        proto_u = np.mean(data["uhist"], axis=0)
        proto_l = np.mean(data["lhist"], axis=0)
        proto_u /= (proto_u.sum() + 1e-12)
        proto_l /= (proto_l.sum() + 1e-12)
        sims = np.dot(embs, proto_emb)
        track_protos[tid] = (proto_emb, proto_u, proto_l)
        track_stats[tid] = {
            "count": len(data["embs"]),
            "mean": float(sims.mean()),
            "min": float(sims.min()),
            "max": float(sims.max()),
            "frame_min": int(min(data["frames"])) if data["frames"] else -1,
            "frame_max": int(max(data["frames"])) if data["frames"] else -1,
            "frames": set(data["frames"]),
        }
    return track_protos, track_stats

def load_folder(folder_path: Path, sess, iname, max_images: Optional[int]):
    folder = Path(folder_path)
    images = list_images(folder, max_images)
    paths = []
    embs = []
    u_hist = []
    l_hist = []
    frames = []
    track_buckets: Dict[int, dict] = defaultdict(lambda: {"embs": [], "uhist": [], "lhist": [], "frames": []})
    for imp in images:
        m = FILENAME_RE.search(imp.stem)
        f = int(m.group(1)) if m else -1
        local_id = int(m.group(2)) if m else -1
        img = cv2.imread(str(imp))
        if img is None:
            continue
        emb = extract_emb(img, sess, iname)
        u, l = extract_color_sig(img)
        paths.append(imp)
        embs.append(emb)
        u_hist.append(u)
        l_hist.append(l)
        frames.append(f)
        track_buckets[local_id]["embs"].append(emb)
        track_buckets[local_id]["uhist"].append(u)
        track_buckets[local_id]["lhist"].append(l)
        track_buckets[local_id]["frames"].append(f)

    if not embs:
        return None

    embs = np.vstack(embs)
    proto_emb = embs.mean(axis=0)
    proto_emb /= (np.linalg.norm(proto_emb) + 1e-12)
    proto_u = np.mean(u_hist, axis=0)
    proto_l = np.mean(l_hist, axis=0)
    proto_u /= (proto_u.sum() + 1e-12)
    proto_l /= (proto_l.sum() + 1e-12)
    track_protos, track_stats = summarize_tracks(track_buckets)
    folder_sims = np.dot(embs, proto_emb)
    return {
        "folder": folder,
        "paths": paths,
        "embs": embs,
        "proto": (proto_emb, proto_u, proto_l),
        "frames": set(frames),
        "track_protos": track_protos,
        "track_stats": track_stats,
        "folder_sim_mean": float(folder_sims.mean()),
        "folder_sim_min": float(folder_sims.min()),
        "folder_sim_max": float(folder_sims.max()),
    }

def load_folder_cached(folder_path: Path, sess, iname, max_images: Optional[int], cache: dict):
    key = (str(folder_path), max_images)
    if key in cache:
        return cache[key]
    data = load_folder(folder_path, sess, iname, max_images)
    cache[key] = data
    return data

In [28]:
# Folder diagnostics to CSV
if not BASE_DIR.exists():
    raise FileNotFoundError(f"BASE_DIR not found: {BASE_DIR}")

sess, iname = load_osnet()
cache = {}

folders = sorted([d for d in BASE_DIR.iterdir() if d.is_dir() and d.name.startswith("person_")])
print(f"Found {len(folders)} folders in {BASE_DIR}")

folder_summary_rows = []
track_summary_rows = []
intra_track_rows = []

for folder in folders:
    data = load_folder_cached(folder, sess, iname, MAX_IMAGES_PER_FOLDER, cache)
    if data is None:
        folder_summary_rows.append({
            "folder": folder.name,
            "path": str(folder),
            "status": "no_images"
        })
        continue

    frame_min = min(data["frames"]) if data["frames"] else -1
    frame_max = max(data["frames"]) if data["frames"] else -1
    overlap_pair_count = 0

    for tid, stats in data["track_stats"].items():
        track_summary_rows.append({
            "folder": folder.name,
            "track_id": tid,
            "image_count": stats["count"],
            "mean_sim": stats["mean"],
            "min_sim": stats["min"],
            "max_sim": stats["max"],
            "frame_min": stats["frame_min"],
            "frame_max": stats["frame_max"],
        })

    track_ids = sorted(data["track_protos"].keys())
    for i in range(len(track_ids)):
        for j in range(i + 1, len(track_ids)):
            t1, t2 = track_ids[i], track_ids[j]
            emb1, u1, l1 = data["track_protos"][t1]
            emb2, u2, l2 = data["track_protos"][t2]
            reid = float(np.dot(emb1, emb2))
            col = color_similarity(u1, l1, u2, l2)
            overlap = len(data["track_stats"][t1]["frames"] & data["track_stats"][t2]["frames"])
            if overlap > 0:
                overlap_pair_count += 1
            merged = (reid >= config.SPLIT_REID_THRESHOLD and col >= config.SPLIT_COLOR_THRESHOLD)
            intra_track_rows.append({
                "folder": folder.name,
                "track_a": t1,
                "track_b": t2,
                "reid_sim": reid,
                "color_sim": col,
                "frame_overlap": overlap,
                "split_reid_threshold": config.SPLIT_REID_THRESHOLD,
                "split_color_threshold": config.SPLIT_COLOR_THRESHOLD,
                "verdict": "MERGED" if merged else "SPLIT"
            })

    folder_summary_rows.append({
        "folder": folder.name,
        "path": str(folder),
        "image_count": len(data["paths"]),
        "track_count": len(data["track_stats"]),
        "frame_min": frame_min,
        "frame_max": frame_max,
        "folder_sim_mean": data["folder_sim_mean"],
        "folder_sim_min": data["folder_sim_min"],
        "folder_sim_max": data["folder_sim_max"],
        "overlap_pair_count": overlap_pair_count,
        "status": "ok"
    })

write_csv(folder_summary_rows, OUTPUT_DIR / "folder_summary.csv")
write_csv(track_summary_rows, OUTPUT_DIR / "track_summary.csv")
write_csv(intra_track_rows, OUTPUT_DIR / "intra_track_pairs.csv")

print("Wrote:")
print(OUTPUT_DIR / "folder_summary.csv")
print(OUTPUT_DIR / "track_summary.csv")
print(OUTPUT_DIR / "intra_track_pairs.csv")

Found 28 folders in outputs/simplified_test_reid_088/refined
Wrote:
outputs/diagnostics_csv_v4/folder_summary.csv
outputs/diagnostics_csv_v4/track_summary.csv
outputs/diagnostics_csv_v4/intra_track_pairs.csv


In [29]:
# Cross-folder compare specs to CSV
compare_summary_rows = []
compare_track_rows = []
compare_top_pairs_rows = []

def choose_merge_thresholds(count_a: int, count_b: int):
    if USE_LARGE_THRESHOLDS and (count_a >= config.LARGE_FOLDER_SIZE or count_b >= config.LARGE_FOLDER_SIZE):
        return config.LARGE_MERGE_REID_THRESHOLD, config.LARGE_MERGE_COLOR_THRESHOLD, "LARGE"
    return config.MERGE_REID_THRESHOLD, config.MERGE_COLOR_THRESHOLD, "NORMAL"

for spec in COMPARE_SPECS:
    for folder1_path, folder2_path in parse_pair_spec(BASE_DIR, spec):
        data1 = load_folder_cached(folder1_path, sess, iname, MAX_IMAGES_PER_FOLDER, cache)
        data2 = load_folder_cached(folder2_path, sess, iname, MAX_IMAGES_PER_FOLDER, cache)

        if data1 is None or data2 is None:
            compare_summary_rows.append({
                "spec": spec,
                "folder1": str(folder1_path),
                "folder2": str(folder2_path),
                "status": "no_images",
            })
            continue

        reid_sim = float(np.dot(data1["proto"][0], data2["proto"][0]))
        col_sim = color_similarity(data1["proto"][1], data1["proto"][2], data2["proto"][1], data2["proto"][2])
        overlap = len(data1["frames"] & data2["frames"])

        reid_thresh, col_thresh, mode = choose_merge_thresholds(len(data1["paths"]), len(data2["paths"]))
        can_merge = True
        reasons = []
        if overlap > config.MAX_SAME_FRAME_OVERLAP:
            can_merge = False
            reasons.append("frame_overlap")
        if reid_sim < reid_thresh:
            can_merge = False
            reasons.append("reid_below")
        if col_sim < col_thresh:
            can_merge = False
            reasons.append("color_below")

        compare_summary_rows.append({
            "spec": spec,
            "folder1": str(folder1_path),
            "folder2": str(folder2_path),
            "folder1_count": len(data1["paths"]),
            "folder2_count": len(data2["paths"]),
            "reid_sim": reid_sim,
            "color_sim": col_sim,
            "frame_overlap": overlap,
            "reid_threshold": reid_thresh,
            "color_threshold": col_thresh,
            "max_frame_overlap": config.MAX_SAME_FRAME_OVERLAP,
            "mode": mode,
            "can_merge": can_merge,
            "reasons": ",".join(reasons) if reasons else "ok",
            "status": "ok",
        })

        track_ids_1 = sorted(data1["track_protos"].keys())
        track_ids_2 = sorted(data2["track_protos"].keys())
        for t1 in track_ids_1:
            emb1, u1, l1 = data1["track_protos"][t1]
            frames_t1 = data1["track_stats"][t1]["frames"]
            for t2 in track_ids_2:
                emb2, u2, l2 = data2["track_protos"][t2]
                frames_t2 = data2["track_stats"][t2]["frames"]
                reid_t = float(np.dot(emb1, emb2))
                col_t = color_similarity(u1, l1, u2, l2)
                ov_t = len(frames_t1 & frames_t2)
                verdict = "MATCH" if (reid_t >= reid_thresh and col_t >= col_thresh) else "NO"
                compare_track_rows.append({
                    "spec": spec,
                    "folder1": str(folder1_path),
                    "folder2": str(folder2_path),
                    "track1": t1,
                    "track2": t2,
                    "reid_sim": reid_t,
                    "color_sim": col_t,
                    "frame_overlap": ov_t,
                    "reid_threshold": reid_thresh,
                    "color_threshold": col_thresh,
                    "verdict": verdict,
                })

        pair_count = len(data1["paths"]) * len(data2["paths"])
        if pair_count > MAX_PAIRWISE:
            compare_top_pairs_rows.append({
                "spec": spec,
                "folder1": str(folder1_path),
                "folder2": str(folder2_path),
                "note": f"skipped_top_pairs: {pair_count} pairs > MAX_PAIRWISE",
            })
        else:
            cos_matrix = np.dot(data1["embs"], data2["embs"].T)
            flat_idx = np.argsort(cos_matrix.ravel())[::-1]
            top_k = min(5, flat_idx.shape[0])
            for idx in flat_idx[:top_k]:
                i = idx // cos_matrix.shape[1]
                j = idx % cos_matrix.shape[1]
                sim = float(cos_matrix[i][j])
                compare_top_pairs_rows.append({
                    "spec": spec,
                    "folder1": str(folder1_path),
                    "folder2": str(folder2_path),
                    "image1": data1["paths"][i].name,
                    "image2": data2["paths"][j].name,
                    "sim": sim,
                })

write_csv(compare_summary_rows, OUTPUT_DIR / "compare_summary.csv")
write_csv(compare_track_rows, OUTPUT_DIR / "compare_track_pairs.csv")
write_csv(compare_top_pairs_rows, OUTPUT_DIR / "compare_top_pairs.csv")

print("Wrote:")
print(OUTPUT_DIR / "compare_summary.csv")
print(OUTPUT_DIR / "compare_track_pairs.csv")
print(OUTPUT_DIR / "compare_top_pairs.csv")

Wrote:
outputs/diagnostics_csv_v4/compare_summary.csv
outputs/diagnostics_csv_v4/compare_track_pairs.csv
outputs/diagnostics_csv_v4/compare_top_pairs.csv
